## Explore Dataset

In [4]:
import geopandas as gpd
import dotenv
import os
import shapely
from addCSOs import calculate_risk_factors_cso

In [5]:
dotenv.load_dotenv()
RIVERS_URL = os.getenv("RIVERS_URL")

## Obtaining data

Keeping all the data in the EPSG 3857 coordinate referencing system (CRS) to work better with distances

### CSO Data

In [6]:
geo_JSON_url = "https://services-eu1.arcgis.com/MSNNjkZ51iVh8yBj/arcgis/rest/services/Northumbrian_Water_Storm_Overflow_Activity_2_view/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
cso: gpd.GeoDataFrame = gpd.read_file(geo_JSON_url)
cso = cso.to_crs(epsg=3857)

### River data

In [8]:
rivers = gpd.read_file(RIVERS_URL)
rivers = rivers.to_crs(epsg=3857)

### Clip to cso bounds

In [7]:
rivers = rivers.clip(cso.total_bounds).explode()

### Catchment Data

In [9]:
cmnt = gpd.read_file(
    "https://environment.data.gov.uk/catchment-planning/OperationalCatchment/3052.geojson")
cmnt.to_crs(cso.crs,inplace=True)

In [10]:
rivers = gpd.clip(rivers, cmnt)
# Remove all rivers that aren't LineStrings after Clipping
rivers = rivers[rivers.geometry.apply(
    lambda x: isinstance(x, shapely.LineString))]
cso = gpd.clip(cso, cmnt)

In [11]:
m = cmnt.explore(highlight=False,
                 popup=False,
                 tooltip=False)
rivers.explore(m=m)
cso.explore(m=m, marker_kwds={"radius":5},color="red")

## Use addCSOs.py

In [22]:
river_points, cso_with_locations = calculate_risk_factors_cso(
    cso,
    rivers,
    cso_df_id_col="Id",
    river_resolution=50,
    max_outfall_dist_from_river=500,
    max_risk_distance=3000,
    include_upstream=False,
    river_directed=True,
)

In [23]:
def simple_risk(csoInfo: list[tuple]):
    tot = 0
    for csoID, dist in csoInfo:
        tot += 3000 - abs(dist)
    return tot

In [24]:
river_points["Risk"] = river_points["csoInfo"].apply(simple_risk)

In [25]:
river_points = river_points[river_points["Risk"] > 0]

## Explore

In [26]:
m = cmnt.explore(m=m,
             highlight = False,
             tooltip = False,
             popup = False,
             )
rivers.explore(
    m=m,
    color="blue",
)
river_points.explore(
    m=m,
    column="Risk",
    cmap="viridis",
    marker_kwds={"radius":5},
    # vmin=20_000,
)
# cso_with_locations.explore(
#     m=m,
#     color="red",
#     marker_kwds={"radius": 8}
# )
cso_with_locations.set_geometry("outfallLocation").explore(
    m=m,
    color="pink",
    marker_kwds={"radius": 8}
)